# private-doc-qa: Colab pipeline

Builds the whole pipeline on **synthetic documents only**: generate documents, ingest and chunk, hybrid search, permission-aware retrieval, cited answers, evaluation.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Use a personal Google account. Never upload real client or employer data here.

Colab sessions are temporary. The code lives in your GitHub repo; this notebook only clones it and calls it.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/azinanigiwu55-dotcom/docreader2.git"  # <- change if you forked it

import os
REPO_DIR = REPO_URL.split("/")[-1].removesuffix(".git")
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR
!pip install -q -e . sentence-transformers accelerate bitsandbytes


## 2. (Optional) Save outputs to Google Drive
Colab wipes files when the session ends. Mounting Drive lets you keep results.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/private-doc-qa-results"
os.makedirs(OUT, exist_ok=True)


## 3. Generate synthetic documents and questions
Fictional firm, fictional clients. Every question has a known answer, so evaluation needs no manual labelling.
`acl.json` says which user groups may read each document.

In [ ]:
from docqa.synth import write_all
print(write_all("data"))
!ls data/docs | head -12
!cat data/acl.json | head -12


## 4. Ingest, chunk and build the hybrid index

In [ ]:
from docqa.ingest import ingest_folder, load_acl
from docqa.index import HybridIndex, SentenceTransformerEmbedder

chunks = ingest_folder("data/docs", load_acl("data/acl.json"))
print(len(chunks), "chunks from", len({c.doc for c in chunks}), "documents")
print(chunks[0])

embedder = SentenceTransformerEmbedder("BAAI/bge-small-en-v1.5")
index = HybridIndex(embedder).build(chunks)
index.save("data/index")


## 5. Search, and check permissions
The same query returns different results depending on who is asking. Restricted chunks are filtered out **before** ranking, so they can never reach the model.

In [ ]:
q = "What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd?"
for groups in (["partners"], ["all"]):
    print("\nUser groups:", groups)
    for h in index.search(q, k=3, user_groups=groups):
        print(f"  {h.score:.4f}  {h.chunk.doc} ({h.chunk.loc})")


## 6. Load a language model
Start with the 3B model (fast, fits the free T4). If you want better answers, switch to 7B in 4-bit; it is slower but still fits.

In [ ]:
from docqa.answer import HFBackend

backend = HFBackend("Qwen/Qwen2.5-3B-Instruct")
# Better quality, slower:
# backend = HFBackend("Qwen/Qwen2.5-7B-Instruct", load_in_4bit=True)


## 7. Ask questions with citations

In [ ]:
from docqa.answer import answer

def ask(question, groups):
    a = answer(question, index, backend, k=4, user_groups=groups,
               user="notebook", audit_path="audit.jsonl")
    print(f"Q: {question}\nGroups: {groups}\nA: {a.text}")
    for c in a.citations:
        print(f"   [{c['n']}] {c['doc']} ({c['loc']})")
    print(f"   found={a.found} reason={a.reason} ({a.latency_s:.1f}s)\n")

ask("What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd in its engagement letter?", ["partners"])
ask("What is the annual audit fee for Karoo Fresh Produce (Pty) Ltd in its engagement letter?", ["all"])   # no access
ask("What is the minimum password length required by the firm?", ["all"])
ask("What is the home address of Aisha Patel?", ["partners"])  # not in the documents


## 8. Run the evaluation
Reports pass rates per question kind: answerable, prompt-injection, unanswerable and forbidden (access control). Use `limit=10` for a quick first run.

In [ ]:
import json
from docqa.eval import run_eval, load_questions, format_report, save_results

questions = load_questions("data/questions.json")
result = run_eval(index, backend, questions, k=4, limit=None)
print(format_report(result))


## 9. Inspect failures and save results
Look at the failures first. They tell you whether to fix retrieval, the prompt, chunking or the model.

In [ ]:
import pandas as pd
df = pd.DataFrame(result["rows"])
display(df[~df["passed"]][["id", "kind", "question", "answer", "reason", "cited"]])

save_results(result, f"{OUT}/results.json")
open(f"{OUT}/results.md", "w").write(format_report(result))
print("Saved to", OUT)


## Next steps
1. Change one thing at a time (chunk size, `k`, model, prompt) and re-run the evaluation. Record the numbers.
2. Copy the final numbers into the README results table.
3. Move the index to Postgres + pgvector once the pipeline is stable (see the README roadmap).